# Geodetic Coordinates to ECI

Given lat/lon/alt of ground station at a particular time, get the ECI coordinates

In [9]:
from astropy.time import Time
from astropy.coordinates import EarthLocation, GCRS
import astropy.units as u

def groundstationECI(lat_deg, lon_deg, alt_m, time):
    station = EarthLocation(lat=lat_deg*u.deg, lon=lon_deg*u.deg, height=alt_m*u.m)
    t = Time(time, scale='utc')
    
    pos, vel = station.get_gcrs_posvel(obstime=t)
	
    r_gs = pos.xyz.to(u.m).value 
    v_gs = vel.xyz.to(u.m/u.s).value
	
    return r_gs, v_gs

## One off time

In [10]:
# Binar GS Geodetic Coords
lat_deg = -32.007
lon_deg = 115.894
alt_m = 50
time = "2026-09-22T12:00:00"

r_gs, v_gs = groundstationECI(lat_deg, lon_deg, alt_m, time)
print(r_gs)

[ 2441901.90112754 -4827329.50363972 -3367363.68610896]


## Multiple times from excel sheet

### Getting time data in correct format

In [ ]:
import pandas as pd

def row_to_iso_string(year, month, day, hour, minute, second):
    """
    Convert separate Y/M/D/H/M/S values (with fractional seconds) 
    into an ISO 8601 datetime string, e.g. '2024-09-13T07:35:05.852424'
    """
    dt_str = (
        f"{int(year):04d}-{int(month):02d}-{int(day):02d}T"
        f"{int(hour):02d}:{int(minute):02d}:{second:09.6f}"
    )
    return dt_str

def convert_row(row):
    return row_to_iso_string(
        row["Year"], row["Month"], row["Day"],
        row["Hour"], row["Minute"], row["Second"]
    )

df = pd.read_excel("FakeDopplerMeasurements.xlsx")

df["ISO Time"] = df.apply(convert_row, axis=1)

print(df)

   Year  Month  Day  Hour  Minute     Second  Doppler Shift (Hz)  Unnamed: 7  \
0  2024      9   13     7      35   5.852424        -9725.117023         NaN   
1  2024      9   13     7      35   7.619209        -9728.591191         NaN   
2  2024      9   13     7      35  16.709893        -9745.148528         NaN   
3  2024      9   13     7      35  32.812891        -9769.477408         NaN   
4  2024      9   13     7      35  37.941555        -9775.998347         NaN   
5  2024      9   13     7      35  48.883421        -9788.101302         NaN   
6  2024      9   13     7      35  54.347516        -9793.275595         NaN   
7  2024      9   13     7      35  54.802551        -9793.681531         NaN   
8  2024      9   13     7      35  57.450410        -9795.969230         NaN   
9  2024      9   13     7      35  57.893312        -9796.339599         NaN   

   Centre Frequency (Hz)                    iso_time  
0              437700000  2024-09-13T07:35:05.852424  
1        

### Doing all Geodetic to ECI conversions

In [ ]:
# Binar GS Geodetic Coords
lat_deg = -32.007
lon_deg = 115.894
alt_m = 50

for time in df["ISO Time"]:
    r_gs, v_gs = groundstationECI(lat_deg, lon_deg, alt_m, time)
    print('\nTIME:', time)
    print('R_GS: ', r_gs)
    print('V_GS: ', v_gs)


TIME: 2024-09-13T07:35:05.852424
R_GS:  [-4022202.60251025 -3632551.277273   -3351356.54150797]
V_GS:  [ 264.87946656 -292.71712886   -0.62329967]

TIME: 2024-09-13T07:35:07.619209
R_GS:  [-4021734.58415509 -3633068.41537293 -3351357.64277654]
V_GS:  [ 264.91717675 -292.68300033   -0.62339138]

TIME: 2024-09-13T07:35:16.709893
R_GS:  [-4019325.42424212 -3635728.30569654 -3351363.31172316]
V_GS:  [ 265.11113844 -292.50732111   -0.62386312]

TIME: 2024-09-13T07:35:32.812891
R_GS:  [-4015053.57584259 -3640436.0430721  -3351373.36406755]
V_GS:  [ 265.45443104 -292.19581213   -0.62469807]

TIME: 2024-09-13T07:35:37.941555
R_GS:  [-4013691.86914428 -3641934.36264545 -3351376.56847356]
V_GS:  [ 265.56368991 -292.09651463   -0.62496382]

TIME: 2024-09-13T07:35:48.883421
R_GS:  [-4010784.83224627 -3645129.28407744 -3351383.40954158]
V_GS:  [ 265.79666656 -291.88452955   -0.62553048]

TIME: 2024-09-13T07:35:54.347516
R_GS:  [-4009332.17639204 -3646723.87954039 -3351386.8281211 ]
V_GS:  [ 265.91